### Get to know your data: Have you recognized issues / assumptions with your data and how do you plan to handle them?

Data granularity. Daily price data gave us fewer than 50k rows, so we switched to 5-minute OHLCV data to increase the number of observations and capture intraday dynamics.
Low-frequency fundamentals and information lag. Fundamentals are only available quarterly. We treat them as piecewise-constant within a quarter and assume a reporting lag (e.g., 45 days) before they become observable to the market. In the merged dataset, each 5-minute bar within a quarter is assigned the most recent fundamental values that are at least 45 days old.

### EDA: Do you have at least 3 meaningful EDA visuals?

Time-series plot of 5-minute close prices – to understand overall trends, volatility clusters and regime changes in the stock over time.

Boxplots of engineered and sentiment features  to inspect skewness, outliers and scale differences, which later motivated winsorization / robust scaling choices.

Correlation heatmap of lagged fundamental features (roe_lag45d, ps_lag45d, de_ratio_lag45d, CAPEI_lag45d, bm_lag45d) – to study multicollinearity among fundamentals and select a subset of five relatively non-redundant but economically meaningful features for modeling

### Modeling: How does the baseline model perform? Comment on the performance. What are the 2-3 models that you will further implement? Why have you chosen these models over other options?


Our baseline is an ARIMAX model using past prices plus a small set of exogenous features. On the test set it achieves:
MSE ≈ 0.0672
MAE ≈ 0.1523
RMSE ≈ 0.2592

Numerically this looks reasonable, but when we plot the predictions the forecast is almost a flat line: the model captures the average price level but fails to react to short-term jumps and volatility. This is consistent with ARIMAX being a linear, low-capacity model that assumes (quasi)-stationarity and struggles with the non-linear, regime-switching behavior in 5-minute intraday data.

Going forward we plan to implement:

Univariate LSTM on intraday prices

Uses a rolling window of 5-minute bars (open/high/low/close, volume, technical features) to predict the next 5-minute close.

LSTMs are designed for sequential dependence and can learn non-linear patterns (e.g., volatility clustering, mean-reversion, momentum) that ARIMAX cannot.

Hybrid LSTM + MLP model (our main model)

LSTM branch: models high-frequency dynamics from intraday data.

MLP branch: takes lower-frequency features (lagged fundamentals, news sentiment features aggregated at daily level).

We then concatenate the LSTM hidden state with the MLP output and add a final dense layer to predict the 5-minute ahead price.

This design lets us combine fast signals (micro-structure / short-term order-flow patterns) and slow signals (valuation, profitability, leverage, sentiment), and allows for non-linear interactions between them. Compared with tree-based models or a pure LSTM, this architecture is better aligned with the data: the two branches can specialize on different time scales but are trained end-to-end.

### Project Management: What’s the plan of action? By when do you plan to complete various stages of this project?

Next week – Sequence models
Implement and tune the univariate LSTM on intraday prices.
Set up the pipeline, early-stopping, and basic hyper-parameter search.

Then, the week after：

Implement the LSTM + MLP hybrid with fundamental and sentiment inputs.

Run systematic experiments comparing ARIMAX, LSTM, and LSTM+MLP on the same splits and metrics (MSE/MAE/RMSE, plus a few plots).

Final week before deadline – Analysis & write-up

Do error analysis (plots of predictions vs. truth, regime breakdowns).

Summarize feature importance

Finalize report and slides